In [ ]:
import json
import random
import re

import gensim.downloader as api
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import f1_score
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.layers import (
    Attention,
    Bidirectional,
    Concatenate,
    Dense,
    Dropout,
    Embedding,
    GlobalAveragePooling1D,
    GlobalMaxPooling1D,
    Input,
    LSTM,
    Multiply,
)
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.text import Tokenizer

DEV_DATA = "./training_data/NLI/dev.csv"
TRAINING_DATA = "./training_data/NLI/train.csv"
TEST_DATA = "./test_data/NLI/test.csv"

random.seed(42)
np.random.seed(42)
tf.random.set_seed(42)

glove_vectors = api.load("glove-wiki-gigaword-300")

## Helper functions

In [ ]:
@tf.keras.utils.register_keras_serializable()
class AbsoluteDifference(tf.keras.layers.Layer):
    def call(self, inputs):
        x1, x2 = inputs
        return tf.abs(x1 - x2)

    def get_config(self):
        return super().get_config()


def clean_text(text):
    text = str(text)
    text = text.replace("\n", " ")
    text = text.replace("\t", " ")
    text = re.sub(r"\s+", " ", text).strip()
    return text


def replace_empty_sequences(sequences, oov_index):
    fixed_sequences = []

    for seq in sequences:
        if len(seq) > 0:
            fixed_sequences.append(seq)
        else:
            fixed_sequences.append([oov_index])

    return fixed_sequences

## Load and Clean data

In [ ]:
train_data_df = pd.read_csv(TRAINING_DATA)
dev_data_df = pd.read_csv(DEV_DATA)
test_data_df = pd.read_csv(TEST_DATA)

for df in (train_data_df, dev_data_df):
    df["premise"] = df["premise"].fillna("").apply(clean_text)
    df["hypothesis"] = df["hypothesis"].fillna("").apply(clean_text)
    df["label"] = df["label"].astype(int)

test_data_df["premise"] = test_data_df["premise"].fillna("").apply(clean_text)
test_data_df["hypothesis"] = test_data_df["hypothesis"].fillna("").apply(clean_text)

train_premises = train_data_df["premise"].tolist()
train_hypotheses = train_data_df["hypothesis"].tolist()
train_labels = train_data_df["label"].astype("float32").to_numpy()

dev_premises = dev_data_df["premise"].tolist()
dev_hypotheses = dev_data_df["hypothesis"].tolist()
dev_labels = dev_data_df["label"].astype("float32").to_numpy()

test_premises = test_data_df["premise"].tolist()
test_hypotheses = test_data_df["hypothesis"].tolist()

## Tokenizer and Padding

In [ ]:
VOCAB_SIZE = 20000
OOV_TOKEN = "<UNK>"

tokenizer = Tokenizer(
    num_words=VOCAB_SIZE,
    oov_token=OOV_TOKEN,
    lower=True,
)

tokenizer.fit_on_texts(train_premises + train_hypotheses)
word_index = tokenizer.word_index

train_premise_seq = tokenizer.texts_to_sequences(train_premises)
train_hypothesis_seq = tokenizer.texts_to_sequences(train_hypotheses)

dev_premise_seq = tokenizer.texts_to_sequences(dev_premises)
dev_hypothesis_seq = tokenizer.texts_to_sequences(dev_hypotheses)

test_premise_seq = tokenizer.texts_to_sequences(test_premises)
test_hypothesis_seq = tokenizer.texts_to_sequences(test_hypotheses)

# get integer ID associated with OOV token
oov_index = tokenizer.word_index[OOV_TOKEN]

# replace empty sequences with a sequence containing the OOV token index
train_premise_seq = replace_empty_sequences(train_premise_seq, oov_index)
train_hypothesis_seq = replace_empty_sequences(train_hypothesis_seq, oov_index)

dev_premise_seq = replace_empty_sequences(dev_premise_seq, oov_index)
dev_hypothesis_seq = replace_empty_sequences(dev_hypothesis_seq, oov_index)

test_premise_seq = replace_empty_sequences(test_premise_seq, oov_index)
test_hypothesis_seq = replace_empty_sequences(test_hypothesis_seq, oov_index)

In [ ]:
# setting premise and hypothesis length to the 95th percentile of training 
MAX_PREMISE_LENGTH = max(
    1,
    int(np.percentile([len(seq) for seq in train_premise_seq], 95)),
)

MAX_HYPOTHESIS_LEN = max(
    1,
    int(np.percentile([len(seq) for seq in train_hypothesis_seq], 95)),
)

# pad every sequence so that they have the same length
train_premise = pad_sequences(
    train_premise_seq,
    maxlen=MAX_PREMISE_LENGTH,
    padding="post",
    truncating="post",
)

train_hypothesis = pad_sequences(
    train_hypothesis_seq,
    maxlen=MAX_HYPOTHESIS_LEN,
    padding="post",
    truncating="post",
)

dev_premise = pad_sequences(
    dev_premise_seq,
    maxlen=MAX_PREMISE_LENGTH,
    padding="post",
    truncating="post",
)

dev_hypothesis = pad_sequences(
    dev_hypothesis_seq,
    maxlen=MAX_HYPOTHESIS_LEN,
    padding="post",
    truncating="post",
)

test_premise = pad_sequences(
    test_premise_seq,
    maxlen=MAX_PREMISE_LENGTH,
    padding="post",
    truncating="post",
)

test_hypothesis = pad_sequences(
    test_hypothesis_seq,
    maxlen=MAX_HYPOTHESIS_LEN,
    padding="post",
    truncating="post",
)

# setting the final embedding vocab size
vocab_size = min(VOCAB_SIZE, len(tokenizer.word_index) + 1)

# MODEL
## Hyper parameters

In [ ]:
EMBEDDING_DIM = 300
LSTM_UNITS = 256
DROPOUT_RATE = 0.3
LEARNING_RATE = 0.001
BATCH_SIZE = 32
EPOCHS = 15

## Building the embedding matrix

In [ ]:
embedding_matrix = np.zeros((vocab_size, EMBEDDING_DIM))

for word, index in word_index.items():
    if index >= vocab_size:
        continue

    if word in glove_vectors:
        embedding_matrix[index] = glove_vectors[word]
    elif index != 0:
        embedding_matrix[index] = np.random.normal(
            loc=0.0,
            scale=0.05,
            size=(EMBEDDING_DIM,),
        )

## embeddings and input
# 2 model inputs 
premise_input = Input(shape=(MAX_PREMISE_LENGTH,), name="premise_input")
hypothesis_input = Input(shape=(MAX_HYPOTHESIS_LEN,), name="hypothesis_input")

# shared embedding layer
# training GloVe does not yield in a higher accuracy!
embedding_layer = Embedding(
    input_dim=vocab_size,
    output_dim=EMBEDDING_DIM,
    weights=[embedding_matrix],
    trainable=False,
    mask_zero=False,
    name="embedding_layer",
)

# converting token IDs to embeddings
premise_embedded = embedding_layer(premise_input)
hypothesis_embedded = embedding_layer(hypothesis_input)

## BiLSTM encoder

In [ ]:
encoder = Bidirectional(
    LSTM(
        LSTM_UNITS,
        return_sequences=True,
        dropout=DROPOUT_RATE,
    ),
    name="encoder",
)

premise_encoded = encoder(premise_embedded)
hypothesis_encoded = encoder(hypothesis_embedded)

## Attention Inspired by Esim
The attention layer aligns each sentence to the other for each word in the premise it finds the most relevant parts of the hypothesis, and vice versa.

In [ ]:
attention_layer = Attention(
    use_scale=True,
    name="attention",
)

premise_aligned = attention_layer([premise_encoded, hypothesis_encoded])
hypothesis_aligned = attention_layer([hypothesis_encoded, premise_encoded])

## Inference - inspired by Esim

Comptuing the difference and the product. Concatenated for an enchanced representation

In [ ]:
premise_diff = AbsoluteDifference(name="premise_diff")(
    [premise_encoded, premise_aligned]
)
hypothesis_diff = AbsoluteDifference(name="hypothesis_diff")(
    [hypothesis_encoded, hypothesis_aligned]
)

premise_mul = Multiply(name="premise_mul")([premise_encoded, premise_aligned])
hypothesis_mul = Multiply(name="hypothesis_mul")([hypothesis_encoded, hypothesis_aligned])

premise_enhanced = Concatenate(name="premise_enhanced")([
    premise_encoded,
    premise_aligned,
    premise_diff,
    premise_mul,
])

hypothesis_enhanced = Concatenate(name="hypothesis_enhanced")([
    hypothesis_encoded,
    hypothesis_aligned,
    hypothesis_diff,
    hypothesis_mul,
])

## Composition biLSTM 
The enhanced representations from the attention step are passed through a second shared BiLSTM

In [ ]:
composition_encoder = Bidirectional(
    LSTM(
        LSTM_UNITS,
        return_sequences=True,
        dropout=DROPOUT_RATE,
    ),
    name="composition_bilstm",
)

premise_composed = composition_encoder(premise_enhanced)
hypothesis_composed = composition_encoder(hypothesis_enhanced)

## Pooling - inspired by Esim
- Average pooling capturing the overall mearning
- Max takes the max value capturing the most important features

In [ ]:
premise_average = GlobalAveragePooling1D()(premise_composed)
premise_max = GlobalMaxPooling1D()(premise_composed)

hypothesis_average = GlobalAveragePooling1D()(hypothesis_composed)
hypothesis_max = GlobalMaxPooling1D()(hypothesis_composed)

# combine the average and max pooled vectors for both premise and hypothesis
premise_vector = Concatenate(name="premise_vector")([premise_average, premise_max])
hypothesis_vector = Concatenate(name="hypothesis_vector")([hypothesis_average, hypothesis_max])

## Final Interaction Features


- Absolute difference between the premise and hypothesis vectors captureing how different the two sentences are overall
- Elementwise product capturing what they have in common

In [ ]:
abs_diff = AbsoluteDifference(name="absolute_difference")(
    [premise_vector, hypothesis_vector]
)

elem_mul = Multiply(name="elementwise_multiply")(
    [premise_vector, hypothesis_vector]
)

merged = Concatenate(name="merged_features")([
    premise_vector,
    hypothesis_vector,
    abs_diff,
    elem_mul,
])

## Classification Layer

In [ ]:
# build the classifier layers
x = Dense(128, activation="relu")(merged)
x = Dropout(DROPOUT_RATE)(x) # prevent overfitting
x = Dense(64, activation="relu")(x)
x = Dropout(DROPOUT_RATE)(x)

# produce the final binary probability output for the 0 and 1 class
output = Dense(1, activation="sigmoid", name="output")(x)

## Compile Model

In [ ]:
model = Model(
    inputs=[premise_input, hypothesis_input],
    outputs=output,
)

# compile the model with binary cross-entropy loss and Adam optimizer
model.compile(
    optimizer=Adam(learning_rate=LEARNING_RATE),
    loss="binary_crossentropy",
    metrics=["accuracy", tf.keras.metrics.AUC(name="auc")],
)

## Training

In [ ]:
# stopping training early if validation loss stops improving
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True,
)

# reduce learning rate if validation loss stops improving
reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5, # half the learning rate when progress stops
    patience=1,
    min_lr=1e-5,
)

history = model.fit(
    x={
        "premise_input": train_premise,
        "hypothesis_input": train_hypothesis,
    },
    y=train_labels,
    validation_data=(
        {
            "premise_input": dev_premise,
            "hypothesis_input": dev_hypothesis,
        },
        dev_labels,
    ),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[early_stop, reduce_lr],
    verbose=1,
)

## Evaluation

In [ ]:
# evlaute model
dev_loss, dev_accuracy, dev_auc = model.evaluate(
    {
        "premise_input": dev_premise,
        "hypothesis_input": dev_hypothesis,
    },
    dev_labels,
    verbose=0,
)

# get predicted probabilities 
dev_probs = model.predict(
    {
        "premise_input": dev_premise,
        "hypothesis_input": dev_hypothesis,
    },
    verbose=0,
).reshape(-1)

# get predicted probabilities for test set
test_probs = model.predict(
    {
        "premise_input": test_premise,
        "hypothesis_input": test_hypothesis,
    },
    verbose=0,
).reshape(-1)

dev_preds = (dev_probs >= 0.5).astype(int)
dev_f1 = f1_score(dev_labels.astype(int), dev_preds)

In [ ]:
# info pringing
print(f"val loss {dev_loss:.4f}")
print(f"val accuracy {dev_accuracy:.4f}")
print(f"val auc {dev_auc:.4f}")
print(f"val f1 {dev_f1:.4f}")

# list of the thresholds to test
thresholds = np.arange(0.10, 0.91, 0.01)
best_threshold = 0.5
best_macro_f1 = -1.0

# try each candidate threshold and keep the one that gives the best macro f1 on dev
for t in thresholds:
    preds = (dev_probs >= t).astype(int)
    score = f1_score(dev_labels.astype(int), preds, average="macro")
    if score > best_macro_f1:
        best_macro_f1 = score
        best_threshold = t

# create the final dev prediction using the best thresholds
dev_preds = (dev_probs >= best_threshold).astype(int)
pd.DataFrame({"label": dev_preds}).to_csv("BILSTM_NLI_dev.csv", index=False)

test_preds = (test_probs >= best_threshold).astype(int)
pd.DataFrame({"label": test_preds}).to_csv("BILSTM_NLI_test.csv", index=False)

## -------
## the following code is for saving the model and important parameters for the demo
## -------

model.save("bilstm_nli_model.keras")

# save 
with open("tokenizer.json", "w") as f:
    f.write(tokenizer.to_json())

# save important parameters to file for demo
with open("model_config.json", "w") as f:
    json.dump(
        {
            "max_premise_length": int(MAX_PREMISE_LENGTH),
            "max_hypothesis_length": int(MAX_HYPOTHESIS_LEN),
            "best_threshold": float(best_threshold),
            "vocab_size": int(VOCAB_SIZE),
            "oov_token": OOV_TOKEN,
        },
        f,
        indent=2,
    )